# Test — Explainer Agent (end-to-end)

Full flow: planner → executor loop → synthesiser

Checks:
- Planner produces `[tool]`-prefixed steps
- Executor queries tool registry per step (inspect which tools were matched)
- Each step produces a raw result
- Synthesiser produces a structured answer in `ui_events`

In [1]:
import sys
sys.path.insert(0, '../app/backend')

from nodes.planner import planner
from nodes.executor import executor
from nodes.synthesiser import synthesiser
from tools import find_tools, registry_summary

In [2]:
# Show what tools are currently available
print("Available tools:\n")
print(registry_summary())

Available tools:

- web_search: Search the web for current or factual information about any topic


In [3]:
state = {
    "task": "How does the transformer attention mechanism work?",
    "messages": [],
    "plan": [],
    "status": "planning",
    "results": [],
    "ui_events": [],
}

In [4]:
# --- Planner ---
state = {**state, **planner(state)}

print(f"Status: {state['status']}")
print(f"Plan ({len(state['plan'])} steps):")
for i, step in enumerate(state['plan'], 1):
    print(f"  {i}. {step}")

Status: executing
Plan (6 steps):
  1. [web_search] What are the key components of the transformer attention mechanism?
  2. [web_search] How is the query, key, and value matrix multiplication performed in attention?
  3. [web_search] What is the role of the softmax function in the attention mechanism?
  4. [web_search] How does the scaled dot-product attention formula work mathematically?
  5. [web_search] What is the difference between self-attention and multi-head attention?
  6. [reason] Synthesize the explanation of how the transformer attention mechanism processes input sequences.


In [5]:
# Verify steps have [tool] prefixes
import re
PREFIX_RE = re.compile(r'^\[[\w_]+\]')
for step in state['plan']:
    assert PREFIX_RE.match(step), f"Step missing [tool] prefix: {step}"
print("All steps have [tool] prefixes ✓")

All steps have [tool] prefixes ✓


In [6]:
# Show which tools the registry would match for each step
print("Tool matching per step:")
for step in state['plan']:
    matched = find_tools(step)
    names = [t.name for t in matched] if matched else ['(none — pure LLM)']
    print(f"  {step[:60]}..." if len(step) > 60 else f"  {step}")
    print(f"    → {names}")

Tool matching per step:
  [web_search] What are the key components of the transformer ...
    → ['tavily_search']
  [web_search] How is the query, key, and value matrix multipl...
    → ['tavily_search']
  [web_search] What is the role of the softmax function in the...
    → ['tavily_search']
  [web_search] How does the scaled dot-product attention formu...
    → ['tavily_search']
  [web_search] What is the difference between self-attention a...
    → ['tavily_search']
  [reason] Synthesize the explanation of how the transformer a...
    → ['(none — pure LLM)']


In [7]:
# --- Executor loop ---
step_num = 0
while state['status'] == 'executing':
    step_num += 1
    current = state['plan'][0]
    print(f"\n=== Step {step_num}: {current} ===")
    state = {**state, **executor(state)}
    print(f"Result (truncated): {state['results'][-1][:300]}...")
    print(f"Status: {state['status']} | Remaining: {len(state['plan'])}")

print(f"\nExecutor done. Status: {state['status']}")


=== Step 1: [web_search] What are the key components of the transformer attention mechanism? ===
Result (truncated): The key components of the transformer attention mechanism are:

1.  **Query (Q), Key (K), and Value (V) Matrices**:
    *   **Query (Q)**: Represents what the model is looking for in the input sequence.
    *   **Key (K)**: Represents the input tokens and allows the model to match against the Query....
Status: executing | Remaining: 5

=== Step 2: [web_search] How is the query, key, and value matrix multiplication performed in attention? ===
Result (truncated): In the attention mechanism (specifically in models like Transformers), the query (Q), key (K), and value (V) matrix multiplication is performed as follows:

### 1. **Matrix Multiplication to Compute Attention Scores**
The first step is to compute the attention scores by multiplying the query matrix ...
Status: executing | Remaining: 4

=== Step 3: [web_search] What is the role of the softmax function in the atten

In [8]:
assert state['status'] == 'synthesising'
assert state['plan'] == []
assert len(state['results']) == step_num
print("Executor assertions passed ✓")

Executor assertions passed ✓


In [9]:
# --- Synthesiser ---
state = {**state, **synthesiser(state)}

print(f"Status: {state['status']}")
print(f"ui_events count: {len(state['ui_events'])}")

Status: done
ui_events count: 1


In [10]:
assert state['status'] == 'done'
assert len(state['ui_events']) == 1
assert state['ui_events'][0]['type'] == 'answer'
print("Synthesiser assertions passed ✓")

Synthesiser assertions passed ✓


In [11]:
# --- Final output ---
from IPython.display import Markdown, display
display(Markdown(state['ui_events'][0]['content']))

# How the Transformer Attention Mechanism Works

The Transformer attention mechanism is the core engine that allows models like BERT and GPT to understand context. Instead of processing words one by one (sequentially), it looks at the entire sentence at once to determine how relevant each word is to every other word. This process is called **Self-Attention**.

Here is a step-by-step breakdown of how it works:

## 1. The Core Ingredients: Query, Key, and Value
Before calculating attention, the model transforms the input words into three different sets of vectors for each word in the sequence:

*   **Query ($Q$):** Represents what the model is *looking for*. For example, if you are looking for the word "Paris," the Query asks, "What matches me?" [source](#)
*   **Key ($K$):** Represents what the model *offers*. Each word has a Key that describes its properties so it can be matched against Queries. [source](#)
*   **Value ($V$):** Represents the *actual content* or information associated with the word. This is the data that gets retrieved once a match is found. [source](#)

> **Analogy:** Imagine a library. The **Query** is your search request ("Find books on history"), the **Key** is the label on the book ("History"), and the **Value** is the actual text inside the book you want to read.

## 2. Calculating Relevance (Dot Product)
The model calculates how well a Query matches a Key using **matrix multiplication** (specifically, the dot product).
*   It compares every Query vector against every Key vector in the sequence.
*   The result is an **Attention Score**, which is a raw number indicating similarity.
*   **Formula:** $Score = Q \cdot K^T$ [source](#)

## 3. Scaling the Scores
Raw dot product scores can become very large as the dimension of the vectors increases. If these numbers get too big, the mathematical function used next (Softmax) breaks down, causing the model to lose the ability to learn.
*   To fix this, the scores are divided by the square root of the vector dimension ($\sqrt{d_k}$).
*   This ensures the variance of the scores remains stable, preventing numerical instability. [source](#)

## 4. Normalizing with Softmax
The scaled scores are passed through the **Softmax** function. This converts the raw scores into **attention weights** (probabilities).
*   Softmax turns the scores into numbers between 0 and 1.
*   Crucially, for any given word (row), the weights sum up to exactly **1**.
*   This allows the model to interpret the scores as a distribution of importance: some words get a high weight (are very relevant), while others get a low weight (are irrelevant). [source](#)

## 5. Weighted Sum of Values
Finally, the model generates the output for each word by taking the **Value** vectors of all other words and combining them based on the attention weights calculated in the previous step.
*   Mathematically: $\text{Output} = \text{Attention Weights} \cdot V$ [source](#)
*   This creates a **context vector** for each word. This vector is a mix of information from the whole sentence, heavily weighted toward the words that are most relevant to the current word.

## 6. Multi-Head Attention
In modern Transformers, the above process is repeated in parallel across multiple "heads."
*   The input is split into smaller parts, and the attention mechanism runs independently for each part.
*   Each head learns to focus on different types of relationships (e.g., one head might focus on grammar/syntax, another on semantic meaning).
*   The results from all heads are combined to create a richer, more comprehensive representation of the text. [source](#)

### Why is this important?
By calculating attention scores for all pairs of words simultaneously, the mechanism captures **long-range dependencies**. It allows the model to understand that "bank" in "river bank" is different from "bank" in "investment bank," even if those words are far apart in the sentence.

---

### Key Takeaways
*   **Three Components:** The mechanism relies on **Query** (what you want), **Key** (what you have), and **Value** (the actual data).
*   **Dynamic Weights:** It calculates a relevance score for every word against every other word, creating a unique context vector for each position.
*   **Softmax Normalization:** The **Softmax** function converts raw similarity scores into probabilities that sum to 1, ensuring stable learning.
*   **Parallel Processing:** Unlike older models that read text sequentially, attention processes the entire sequence at once, enabling the capture of long-range dependencies.
*   **Multi-Head Architecture:** Running attention in parallel across multiple "heads" allows the model to capture diverse types of relationships simultaneously.